# DataTrust — Modelo predictivo: original vs. tratado

## Contexto

Los notebooks 01-05 auditaron las tres tablas, consolidaron 14 hallazgos en una tabla de gobernanza, y ejecutaron el tratamiento de curación. Este notebook responde al **objetivo 5** del proyecto: comparar el rendimiento de un modelo predictivo entrenado con datos originales (`raw`) frente al mismo modelo entrenado con datos tratados (`curado`).

## Objetivo de este notebook

Entrenar el **mismo modelo** (mismo algoritmo, mismo split, misma semilla) en tres versiones de los datos:

1. **CRUDO**: `data/airbnb_bcn/listings.csv`, sin las decisiones de gobernanza del notebook 05.
2. **TRATADO — criterio A**: `data/airbnb_bcn_curado/listings.csv`, el dataset curado tal como lo produjo el notebook 05 (imputación de `bedrooms`/`bathrooms`/`beds` por mediana de `accommodates`).
3. **TRATADO — criterio B**: el mismo dataset curado, pero reimputando `bedrooms`/`bathrooms`/`beds` por mediana de `neighbourhood_cleansed` en vez de por capacidad — un **chequeo de sensibilidad** (ver "Limitaciones metodológicas" en la ficha maestra): las decisiones de curación están justificadas, pero no se pueden validar contra un valor real conocido, así que se comprueba si la conclusión depende de cuál de dos criterios razonables se use.

No se asume de antemano que el modelo tratado tiene que ganar. Cualquier resultado (mejora, empeora, casi no cambia) es válido y se reporta tal cual, junto con la explicación de por qué ocurre.

## Variables (definidas en la ficha maestra, no se vuelven a discutir aquí)

- **Target**: `estimated_revenue_l365d` (regresión). Estimación de Inside Airbnb, no ingreso real observado — limitación ya declarada en la ficha maestra.
- **Predictoras permitidas (sin fuga)**: `room_type`, `property_type`, `neighbourhood_cleansed`, `accommodates`, `bedrooms`, `bathrooms`, `beds`, `host_listings_count`, `calculated_host_listings_count`, `host_is_superhost`, `amenities` (parseada a un conteo numérico).
- **Excluidas por fuga de información**: `price`, `estimated_occupancy_l365d`, `number_of_reviews` (y variantes), `minimum_nights` — forman parte de la fórmula con la que Inside Airbnb calcula el target, usarlas sería hacer trampa al modelo.

**Nota sobre `host_listings_count` vs. `calculated_host_listings_count`**: se incluyen las dos. No son redundantes (correlación 0,93, no 1,0): `calculated_host_listings_count` mide la escala del anfitrión específicamente en Barcelona (sin nulos, calculada por Inside Airbnb sobre este snapshot), `host_listings_count` es un conteo global de toda la plataforma (87 nulos, dato de perfil). Esta es también la columna corregida para definir `host_profesional` en el notebook 05, tras verificar contra el diccionario de datos oficial de Inside Airbnb que `host_listings_count` no está limitada a Barcelona. Se dejan ambas como predictoras para que la importancia de variables muestre cuál aporta más señal, en vez de elegir una de antemano.

**Algoritmo**: `RandomForestRegressor` (200 árboles, semilla 42), igual que en el preliminar `check_impacto_preliminar.py` que ya validó este enfoque. Mismo algoritmo y mismo split (80/20, semilla 42) en las tres versiones, para que cualquier diferencia en las métricas venga del tratamiento de los datos, no de un cambio de modelo.

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
listings_raw = pd.read_csv("../data/airbnb_bcn/listings.csv", low_memory=False)
listings_curado = pd.read_csv("../data/airbnb_bcn_curado/listings.csv", low_memory=False)

print("listings_raw:   ", listings_raw.shape)
print("listings_curado:", listings_curado.shape)

listings_raw:    (15293, 90)
listings_curado: (15293, 91)


## 1. Preparación común: funciones de features y de evaluación

Las mismas funciones se usan para las tres versiones del dataset, para garantizar que la única diferencia entre ellas sea el tratamiento de los datos, no la forma de construir las variables.

**Nota sobre `amenities`**: en la versión CRUDO se usa un proxy burdo (longitud del texto), simulando lo que haría un análisis rápido sin parsear de verdad el campo. En las versiones TRATADO se parsea correctamente como lista JSON para contar los amenities reales. Esta diferencia es intencional: parsear bien un campo de texto estructurado es, en sí mismo, una decisión de calidad de datos, no solo una cuestión de imputación de nulos.

In [3]:
CATS = ["room_type", "property_type", "neighbourhood_cleansed"]
NUMS = ["accommodates", "bedrooms", "bathrooms", "beds", "host_listings_count", "calculated_host_listings_count"]


def amenities_count_crudo(serie):
    """Proxy burdo: longitud del texto, sin parsear la lista real."""
    return serie.astype(str).str.len()


def amenities_count_real(valor):
    """Parseo correcto: numero de amenities en la lista JSON."""
    try:
        return len(json.loads(valor))
    except Exception:
        return np.nan


def construye_features(df, amenities_fn, es_crudo):
    X = df[CATS + NUMS].copy()
    X["amenities_count"] = amenities_fn(df["amenities"]) if es_crudo else df["amenities"].apply(amenities_fn)
    X["host_is_superhost"] = df["host_is_superhost"].fillna("f")
    return pd.get_dummies(X, columns=CATS + ["host_is_superhost"], drop_first=True)


def evaluar(X, y, nombre):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    modelo = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    modelo.fit(Xtr, ytr)
    pred = modelo.predict(Xte)
    mae = mean_absolute_error(yte, pred)
    rmse = mean_squared_error(yte, pred) ** 0.5
    r2 = r2_score(yte, pred)
    print(f"{nombre:16s} n={len(y):6d}  n_features={X.shape[1]:3d}  MAE={mae:8.1f}  RMSE={rmse:8.1f}  R2={r2:.3f}")
    return {"nombre": nombre, "mae": mae, "rmse": rmse, "r2": r2, "n": len(y), "modelo": modelo, "cols": X.columns}

## 2. Versión CRUDO

**Tratamiento de nulos**: sentinela `-1` para `bedrooms`/`bathrooms`/`beds` (un relleno arbitrario, no una imputación razonada — simula lo que haría un análisis que no audita el dato). `host_listings_count` se rellena con 1 solo para evitar que el modelo falle por nulos, no es una decisión de negocio.

**Tratamiento del target**: se descartan únicamente las filas sin dato (`dropna`), sin investigar los ceros. Esto significa que los 1.895 anuncios con ingreso "no estimable" (ver ficha maestra) se tratan como si tuvieran un ingreso real de 0 €, exactamente el problema que ya se diagnosticó en el caso de negocio.

In [4]:
A = listings_raw.copy()
A["bedrooms"] = A["bedrooms"].fillna(-1)
A["bathrooms"] = A["bathrooms"].fillna(-1)
A["beds"] = A["beds"].fillna(-1)
A["host_listings_count"] = A["host_listings_count"].fillna(1)

df_crudo = A.dropna(subset=["estimated_revenue_l365d"]).copy()
y_crudo = df_crudo["estimated_revenue_l365d"]
X_crudo = construye_features(df_crudo, amenities_count_crudo, es_crudo=True)

print("Filas usadas (con target no nulo):", len(df_crudo))
res_crudo = evaluar(X_crudo, y_crudo, "CRUDO")

Filas usadas (con target no nulo): 13355


CRUDO            n= 13355  n_features=125  MAE= 13309.7  RMSE= 20908.6  R2=0.470


**Resultado**: 13.355 filas (todas las que tienen `estimated_revenue_l365d` no nulo), sin excluir los casos "no estimable". Esta es la versión que un análisis sin auditar produciría.

## 3. Versión TRATADO — criterio A (notebook 05)

**Tratamiento de nulos**: `bedrooms`/`bathrooms`/`beds` ya vienen imputados por mediana de `accommodates` (hecho en el notebook 05). `amenities` se parsea correctamente.

**Tratamiento del target**: se excluyen los anuncios marcados `revenue_no_estimable` (la doble evidencia: sin reseñas en el último año + estancia mínima ≥28 noches), no se tratan como ingreso real de 0 €. Esta es la diferencia central con la versión CRUDO: no es solo "imputar mejor los nulos de las predictoras", es tratar correctamente el propio target.

In [5]:
B = listings_curado.copy()
B["host_listings_count"] = B["host_listings_count"].fillna(1)

df_tratado_a = B[B["estimated_revenue_l365d"].notna() & ~B["revenue_no_estimable"]].copy()
y_tratado_a = df_tratado_a["estimated_revenue_l365d"]
X_tratado_a = construye_features(df_tratado_a, amenities_count_real, es_crudo=False)

print("Filas usadas (excluyendo revenue_no_estimable):", len(df_tratado_a))
res_tratado_a = evaluar(X_tratado_a, y_tratado_a, "TRATADO-A")

Filas usadas (excluyendo revenue_no_estimable): 10669


TRATADO-A        n= 10669  n_features=122  MAE= 15215.9  RMSE= 23056.6  R2=0.439


**Resultado**: 10.669 filas, 2.686 menos que en CRUDO — exactamente los anuncios marcados como "no estimable" (con la columna corregida `number_of_reviews_ltm`, ver notebook 05) que se excluyen en vez de tratarse como ingreso real cero.

## 4. Comparación CRUDO vs. TRATADO (objetivo 5 del proyecto)

In [6]:
print(f"{'Version':10s} {'n':>8s} {'MAE':>10s} {'RMSE':>10s} {'R2':>8s}")
for r in [res_crudo, res_tratado_a]:
    print(f"{r['nombre']:10s} {r['n']:8d} {r['mae']:10.1f} {r['rmse']:10.1f} {r['r2']:8.3f}")

print()
print(f"Diferencia MAE  (crudo - tratado-A): {res_crudo['mae'] - res_tratado_a['mae']:.1f}")
print(f"Diferencia RMSE (crudo - tratado-A): {res_crudo['rmse'] - res_tratado_a['rmse']:.1f}")
print(f"Diferencia R2   (tratado-A - crudo): {res_tratado_a['r2'] - res_crudo['r2']:.3f}")

Version           n        MAE       RMSE       R2
CRUDO         13355    13309.7    20908.6    0.470
TRATADO-A     10669    15215.9    23056.6    0.439

Diferencia MAE  (crudo - tratado-A): -1906.2
Diferencia RMSE (crudo - tratado-A): -2148.0
Diferencia R2   (tratado-A - crudo): -0.031


**Resultado**: el modelo TRATADO-A tiene un **MAE peor** que el CRUDO (15.215,9 € vs. 13.309,7 €, +1.906,2 €) y un **R² algo más bajo** (0,439 vs. 0,470, −0,031).

**Por qué el tratado no "gana", y por qué eso no significa que el tratamiento esté mal**: el modelo CRUDO se evalúa sobre una población que incluye 2.686 anuncios con ingreso 0 € "artificial" (el problema de cálculo diagnosticado en el caso de negocio, con la columna corregida `number_of_reviews_ltm`). Predecir cerca de 0 € para esos casos es una tarea fácil para cualquier modelo, lo que infla artificialmente su precisión aparente — y cuantos más de esos casos fáciles tenga la población evaluada, mayor es esa inflación. El modelo TRATADO-A se evalúa sobre una población sin esos 2.686 casos triviales — una tarea objetivamente más difícil y más honesta, porque ya no incluye puntos fáciles de acertar que distorsionan la métrica.

En otras palabras: el CRUDO no es un mejor modelo, es un modelo evaluado con una vara más fácil. Esto es exactamente el tipo de hallazgo "tentador pero incompleto" que advierte la ficha maestra: una lectura rápida de "el MAE empeoró, el tratamiento no sirvió" sería engañosa sin esta explicación. La brecha es incluso mayor que en una versión preliminar de este análisis (que excluía solo 1.895 casos con una columna menos precisa): con más casos "fáciles" correctamente identificados y excluidos, la diferencia de dificultad entre ambas poblaciones se hace más visible, no menos — una señal más de que la explicación es consistente, no una coincidencia puntual.

## 5. Chequeo de sensibilidad — criterio B (reimputar por barrio en vez de por capacidad)

**Motivación** (ver "Limitaciones metodológicas" en la ficha maestra): las decisiones de curación están justificadas, pero no se pueden validar contra un valor real conocido — no hay forma de probar que imputar por `accommodates` sea "la verdad". Se reimputan `bedrooms`/`bathrooms`/`beds` por mediana de `neighbourhood_cleansed` (criterio B) y se comprueba si la conclusión del modelo cambia mucho frente al criterio A. Se mantiene la mediana como estadístico en ambos criterios (no se cambia también a la media), para aislar el efecto de la variable de agrupación y no mezclar dos cambios a la vez.

In [7]:
C = listings_curado.copy()
C["host_listings_count"] = C["host_listings_count"].fillna(1)

# Deshacer la imputacion por accommodates (donde *_imputado es True) y reimputar por barrio
for col in ["bedrooms", "bathrooms", "beds"]:
    C.loc[C[f"{col}_imputado"], col] = np.nan
    mediana_barrio = C.groupby("neighbourhood_cleansed")[col].transform("median")
    mediana_global = C[col].median()
    C[col] = C[col].fillna(mediana_barrio).fillna(mediana_global)

df_tratado_b = C[C["estimated_revenue_l365d"].notna() & ~C["revenue_no_estimable"]].copy()
y_tratado_b = df_tratado_b["estimated_revenue_l365d"]
X_tratado_b = construye_features(df_tratado_b, amenities_count_real, es_crudo=False)

print("Filas usadas (mismas que TRATADO-A, cambia solo el criterio de imputacion):", len(df_tratado_b))
res_tratado_b = evaluar(X_tratado_b, y_tratado_b, "TRATADO-B")

Filas usadas (mismas que TRATADO-A, cambia solo el criterio de imputacion): 10669


TRATADO-B        n= 10669  n_features=122  MAE= 15205.4  RMSE= 23036.9  R2=0.440


In [8]:
print(f"{'Version':10s} {'n':>8s} {'MAE':>10s} {'RMSE':>10s} {'R2':>8s}")
for r in [res_tratado_a, res_tratado_b]:
    print(f"{r['nombre']:10s} {r['n']:8d} {r['mae']:10.1f} {r['rmse']:10.1f} {r['r2']:8.3f}")

print()
print(f"Diferencia MAE  (criterio A - criterio B): {res_tratado_a['mae'] - res_tratado_b['mae']:.1f}")
print(f"Diferencia RMSE (criterio A - criterio B): {res_tratado_a['rmse'] - res_tratado_b['rmse']:.1f}")
print(f"Diferencia R2   (criterio A - criterio B): {res_tratado_a['r2'] - res_tratado_b['r2']:.3f}")
print(f"Diferencia relativa de MAE: {(res_tratado_a['mae'] - res_tratado_b['mae']) / res_tratado_a['mae'] * 100:.2f}%")

Version           n        MAE       RMSE       R2
TRATADO-A     10669    15215.9    23056.6    0.439
TRATADO-B     10669    15205.4    23036.9    0.440

Diferencia MAE  (criterio A - criterio B): 10.5
Diferencia RMSE (criterio A - criterio B): 19.7
Diferencia R2   (criterio A - criterio B): -0.001
Diferencia relativa de MAE: 0.07%


**Resultado**: la diferencia entre los dos criterios de imputación es mínima — MAE difiere en apenas 10,5 € (0,07% relativo), RMSE en 19,7 € (~0,09%), y R² en 0,001. Las 10.669 filas usadas son idénticas entre criterio A y B (el filtro del target no cambia, solo la imputación de tres predictoras).

**Conclusión del chequeo de sensibilidad**: la conclusión del modelo **no depende** de cuál de los dos criterios razonables de imputación se use — y la estabilidad es aún mayor que en una versión preliminar de este chequeo. Esto no demuestra que la curación sea "la verdad" (eso sigue sin poder probarse sin un valor real conocido), pero sí es evidencia de que no es una elección arbitraria o frágil — el resultado es estable ante al menos dos formas razonables de resolver el mismo problema de nulos.

## 6. Importancia de variables

In [9]:
print("Top 8 variables — CRUDO")
imp_crudo = pd.Series(res_crudo["modelo"].feature_importances_, index=res_crudo["cols"]).sort_values(ascending=False)
print(imp_crudo.head(8))

print()
print("Top 8 variables — TRATADO-A")
imp_tratado_a = pd.Series(res_tratado_a["modelo"].feature_importances_, index=res_tratado_a["cols"]).sort_values(ascending=False)
print(imp_tratado_a.head(8))

print()
print("Top 8 variables — TRATADO-B")
imp_tratado_b = pd.Series(res_tratado_b["modelo"].feature_importances_, index=res_tratado_b["cols"]).sort_values(ascending=False)
print(imp_tratado_b.head(8))

Top 8 variables — CRUDO
amenities_count                   0.201680
host_listings_count               0.132026
host_is_superhost_t               0.107333
accommodates                      0.105208
beds                              0.103485
calculated_host_listings_count    0.067628
bathrooms                         0.048920
bedrooms                          0.040290
dtype: float64

Top 8 variables — TRATADO-A
amenities_count                   0.176955
host_listings_count               0.146300
accommodates                      0.129843
host_is_superhost_t               0.080389
calculated_host_listings_count    0.070018
beds                              0.068183
bathrooms                         0.066426
bedrooms                          0.050389
dtype: float64

Top 8 variables — TRATADO-B


amenities_count                   0.173663
accommodates                      0.151950
host_listings_count               0.146980
host_is_superhost_t               0.081463
calculated_host_listings_count    0.070223
bathrooms                         0.063509
beds                              0.055149
bedrooms                          0.047042
dtype: float64


**Resultado**: las mismas 8 variables dominan en las tres versiones — `amenities_count`, `host_listings_count`, `calculated_host_listings_count`, `host_is_superhost`, `accommodates`, `beds`, `bathrooms` y `bedrooms` — sin que ninguna variable geográfica concreta entre en el top 8 (a diferencia de una versión preliminar de este análisis). El orden relativo cambia ligeramente entre versiones, pero no hay ninguna variable que aparezca como importante en una versión y desaparezca en otra — otra señal de estabilidad, consistente con el chequeo de sensibilidad de la sección anterior.

**Sobre las dos variables de conteo de anuncios**: `host_listings_count` (escala global del anfitrión) resulta más importante para el modelo que `calculated_host_listings_count` (escala específica en Barcelona) en las tres versiones — el modelo encuentra más señal predictiva en el conteo global. Esto no contradice la decisión de usar `calculated_host_listings_count` para definir `host_profesional` (esa es una decisión de **definición/segmentación para el caso de negocio**, donde importa medir la escala en el mercado que se estudia), es un hallazgo complementario y distinto: para **predecir ingresos**, la escala global del anfitrión aporta más información que su escala local. Ambas cosas pueden ser ciertas a la vez.

Nota sobre el número de columnas (`n_features`): varía ligeramente entre versiones (125 en CRUDO, 122 en TRATADO) porque el one-hot encoding genera una columna por cada categoría presente en el subconjunto de filas usado, y los subconjuntos no son idénticos (CRUDO incluye los 2.686 anuncios "no estimable" que TRATADO excluye). No es una inconsistencia en el método, es un efecto esperado de filtrar filas distintas.

## Resumen — Notebook 06

| Versión | Filas | MAE | RMSE | R² |
|---|---|---|---|---|
| CRUDO | 13.355 | 13.309,7 € | 20.908,6 € | 0,470 |
| TRATADO — criterio A (capacidad) | 10.669 | 15.215,9 € | 23.056,6 € | 0,439 |
| TRATADO — criterio B (barrio) | 10.669 | 15.205,4 € | 23.036,9 € | 0,440 |

**Hallazgo principal**: el modelo tratado no "gana" en MAE ni en R² frente al crudo, pero no porque el tratamiento sea peor — porque el crudo se beneficia artificialmente de 2.686 casos de ingreso "0 €" fáciles de predecir que en realidad son un problema de cálculo (columna `number_of_reviews_ltm`, corregida en el notebook 05), no un ingreso real. Tal como establece la ficha maestra, **cualquier resultado es válido y se reporta**: en este caso, el resultado no es "el modelo tratado es mejor", es "el modelo tratado se evalúa de forma más honesta, sobre un problema objetivamente más difícil, y esa brecha es más visible cuantos más casos fáciles se excluyen correctamente".

**Chequeo de sensibilidad**: la elección del criterio de imputación (capacidad vs. barrio) cambia el resultado en menos del 0,1% — la conclusión no depende de esa decisión concreta, lo que respalda que la curación no es arbitraria, dentro de la limitación ya declarada de que no puede validarse contra un valor real conocido.

**Corrección metodológica aplicada en esta versión**: tras una revisión a fondo (ver ficha maestra, sección "Traducción a impacto monetario"), se corrigieron dos variables heredadas de un análisis anterior: `host_profesional` ahora usa `calculated_host_listings_count` (escala del anfitrión en Barcelona, no global) y `revenue_no_estimable` usa `number_of_reviews_ltm` (reseñas del último año, alineadas con la ventana temporal del target) en vez de las columnas menos precisas usadas antes. Ninguna de las dos correcciones debilitó las conclusiones del proyecto.

Este notebook cierra la parte técnica del proyecto (objetivos 1-5). Quedan pendientes el dashboard de Power BI, el informe en PDF y la presentación para la defensa.